# cuda-engine — Colab quickstart

Synthesize a verified CUDA kernel from a natural-language prompt + a PyTorch reference, end-to-end on a Colab A100.

**Prerequisites:**
- Colab Pro with a GPU runtime (Runtime → Change runtime type → A100 GPU)
- An Anthropic API key set as a Colab secret named `ANTHROPIC_API_KEY` (key icon in the sidebar)

Total wall time: ~3–8 minutes for a happy-path run. Cost: ~$0.10–0.40.

## 1. Verify GPU + CUDA toolchain

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,compute_cap --format=csv,noheader
!which nvcc && nvcc --version | tail -1
!which ncu && ncu --version | head -1

## 2. Install cuda-engine

In [ ]:
# Once cuda-engine is published, this becomes a one-liner:
# !pip install --quiet cuda-engine
# For now, install from the source checkout:
%cd /content
!rm -rf Cuda-Engine
!git clone --depth 1 https://github.com/shivnarainms22/Cuda-Engine.git
%cd Cuda-Engine
!pip install -e . --quiet

## 3. Set the Anthropic API key

In [ ]:
import os

from google.colab import userdata

os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
assert os.environ['ANTHROPIC_API_KEY'].startswith('sk-'), 'set ANTHROPIC_API_KEY in Colab secrets'

## 4. Define a reference function

A simple fp32 RMSNorm. The reference is what cuda-engine compares the generated kernel against for correctness.

In [ ]:
import torch


def rms_norm(x):
    return x * (x.float().pow(2).mean(dim=-1, keepdim=True) + 1e-5).rsqrt().to(x.dtype)

# Sanity-check the reference runs on CUDA before invoking synthesize().
x_test = torch.randn(2, 64, device='cuda', dtype=torch.float32)
_ = rms_norm(x_test)
print('reference function works')

## 5. Synthesize the kernel

This is the main call. Watch the printed output — each stage logs its progress.

In [ ]:
from cuda_engine import SynthesisConfig, synthesize
from cuda_engine.config import RetryBudgets

result = synthesize(
    prompt='Compute RMSNorm over the last dim of a 2D tensor: x * rsqrt(mean(x**2, dim=-1, keepdim=True) + 1e-5).',
    reference=rms_norm,
    target='sm_80',
    config=SynthesisConfig(
        retry_budgets=RetryBudgets(codegen=3, correctness=3, performance=2),
        opus_retry_budget_performance=1,
    ),
)

print(f'passed: {result.passed}')
print(f'run_id: {result.run_id}')
print(f'artifacts: {result.artifacts_dir}')

## 6. Inspect the result

In [ ]:
if result.passed:
    print(f'Correctness: {result.correctness.passed}')
    if result.performance and result.performance.speedup_vs_torch_compile is not None:
        print(f'Speedup vs torch.compile: {result.performance.speedup_vs_torch_compile:.2f}x')
        print(f'Below target: {result.performance.below_target}')
    for trace in result.report.stage_traces:
        print(f'  {trace.stage_name:14}: {"ok" if trace.succeeded else "failed":7} '
              f'attempts={trace.attempts} model={trace.model_used} '
              f'tokens={trace.tokens_in}/{trace.tokens_out}')
else:
    print(f'Failed at stage {result.failed_stage}: {result.failure_reason}')

## 7. Read the generated kernel

In [ ]:
from pathlib import Path

kernel_path = Path(result.artifacts_dir) / 'stage5_polish' / 'final' / 'kernel.cu'
if not kernel_path.exists():
    kernel_path = Path(result.artifacts_dir) / 'stage2_codegen' / 'final' / 'kernel.cu'
print(kernel_path.read_text())

## 8. Use the kernel as a Torch op (optional)

The compiled `.so` is loadable via `torch.ops.load_library` and exposes a `cuda_engine.forward` op. The `result.kernel_callable` field provides a thin wrapper when synthesis succeeds.

In [ ]:
if result.passed and result.kernel_callable is not None:
    out = result.kernel_callable(x_test)
    expected = rms_norm(x_test)
    max_err = (out - expected).abs().max().item()
    print(f'max abs error vs reference: {max_err:.3e}')
else:
    print('No callable available — load the .so manually via torch.ops.load_library if needed.')

## Next steps

- Try different prompts and references. The `evals/internal/` directory has 30 working examples covering elementwise ops, reductions, fused kernels.
- Run the full eval suite: `!cuda-engine eval --suite internal --out evals/results/$(date +%Y-%m-%d) --resume`
- Read the design doc: `docs/superpowers/specs/2026-04-26-cuda-synthesis-engine-design.md`
- Cost / privacy / scope: `docs/cost.md`, `docs/privacy.md`, `README.md`